# Data Preparation: 30-Class Fruit Classification (FIDS30)

Download the FIDS30 dataset and split into Training/Validation/Test sets.

In [ ]:
import urllib.request
import zipfile
from pathlib import Path

DATA_URL = "https://data.vicos.si/datasets/FIDS30/FIDS30.zip"
ZIP_PATH = Path("FIDS30.zip")
EXTRACT_DIR = Path("data")

if not (EXTRACT_DIR / "FIDS30").exists():
    if not ZIP_PATH.exists():
        print(f"Downloading FIDS30 dataset...")
        urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
        print(f"Downloaded: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e6:.1f} MB)")

    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print(f"Extracted to {EXTRACT_DIR}/")
else:
    print("FIDS30 dataset already present, skipping download.")

# Show discovered classes
source = EXTRACT_DIR / "FIDS30"
classes = sorted([d.name for d in source.iterdir() if d.is_dir()])
print(f"\nFound {len(classes)} classes: {classes}")

In [ ]:
import os
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split

SOURCE_BASE = Path("data/FIDS30")
TARGET_BASE = Path("PrepData")

# Discover all classes and collect image paths with labels
all_images = []
all_labels = []
classes = sorted([d.name for d in SOURCE_BASE.iterdir() if d.is_dir()])

for cls_name in classes:
    cls_dir = SOURCE_BASE / cls_name
    imgs = [p for p in cls_dir.glob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]]
    all_images.extend(imgs)
    all_labels.extend([cls_name] * len(imgs))

print(f"Total images: {len(all_images)} across {len(classes)} classes")

# Split: 60% train, 20% val, 20% test (stratified)
train_imgs, temp_imgs, train_labels, temp_labels = train_test_split(
    all_images, all_labels, test_size=0.4, stratify=all_labels, random_state=42
)
val_imgs, test_imgs, val_labels, test_labels = train_test_split(
    temp_imgs, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42
)

splits = {
    "Training": (train_imgs, train_labels),
    "Validation": (val_imgs, val_labels),
    "Test": (test_imgs, test_labels),
}

# Clean up existing PrepData
if TARGET_BASE.exists():
    print(f"Removing existing directory: {TARGET_BASE}")
    shutil.rmtree(TARGET_BASE)

# Symlink images into PrepData in ImageFolder format
for split_name, (images, labels) in splits.items():
    for img_path, cls_name in zip(images, labels):
        dest_dir = TARGET_BASE / split_name / cls_name
        dest_dir.mkdir(parents=True, exist_ok=True)
        os.symlink(img_path.absolute(), dest_dir / img_path.name)
    print(f"  {split_name}: {len(images)} images")

print("\nDone! Dataset saved to PrepData/")